# Self-Similar Hyperbolic Tilings

This notebook explores hyperbolic tilings with self-similar structure based on the Margulis-Mozes construction.

**Contents:**
1. Horocycle Scene - Horizontal horocycles at multiple scale levels with rays
2. Bumps Scene with Polygon - Hierarchical bumps (geodesic segments), rays, and a single polygon
3. Debug: Single Polygon - Verify polygon placement
4. Full Tiling with Bumps - Complete scene with all tiles
5. Pure Polygon Tiling - Clean tiling without bumps/rays
6. 3-Coloring Algorithm - Demonstration that the tiling is 3-colorable

## 1. Horocycle Scene

Horizontal horocycles at heights y = m^k for multiple scale levels, with rays from interior kink points connecting the hierarchy. This is the simplest representation of the self-similar structure.

In the upper half-plane, horocycles tangent to infinity appear as horizontal lines. The vertices of the "bumps" in Section 2 all lie on these horocycles.

In [ ]:
import math
from hyperbolic_svg import HyperbolicScene, make_hyperbolic_polygon_from_params
from IPython.display import SVG, display

# =============================================================================
# PARAMETERS - Adjust these to customize the scene
# =============================================================================

m = 2              # Scaling factor (branching ratio)
x_range = 1000     # Horizontal extent of the scene (before offset)
OFFSET = -x_range/2 - 1/8  # Translate scene by this real value
MIN_LEVEL = -5     # Lowest level (finest detail, closest to boundary)
MAX_LEVEL = 9      # Highest level (coarsest detail, closest to center)
hwidth = 0.08      # Hyperbolic line width (same for all elements)
ray_color = '#555555'  # Color for rays from interior kinks
WIDTH = 600        # SVG width in pixels
HEIGHT = 600       # SVG height in pixels

# =============================================================================
# COLOR FUNCTION
# =============================================================================

def level_color(level, min_lvl, max_lvl):
    """Generate a color based on level (hue rotation from red to purple)."""
    t = (level - min_lvl) / (max_lvl - min_lvl) if max_lvl != min_lvl else 0.5
    hue = t * 300
    if hue < 60:
        r, g, b = 255, int(hue * 255 / 60), 0
    elif hue < 120:
        r, g, b = int((120 - hue) * 255 / 60), 255, 0
    elif hue < 180:
        r, g, b = 0, 255, int((hue - 120) * 255 / 60)
    elif hue < 240:
        r, g, b = 0, int((240 - hue) * 255 / 60), 255
    else:
        r, g, b = int((hue - 240) * 255 / 60), 0, 255
    return f'#{r:02x}{g:02x}{b:02x}'

# =============================================================================
# BUILD THE SCENE
# =============================================================================

scene = HyperbolicScene()

# --- Draw horizontal horocycles at each level ---
print(f"Building horocycle scene with m={m}, levels {MIN_LEVEL} to {MAX_LEVEL}")
print("\nHorocycles:")

for level in range(MIN_LEVEL, MAX_LEVEL + 1):
    height = m ** level
    color = level_color(level, MIN_LEVEL, MAX_LEVEL)
    scene.add_horizontal_horocycle(height, hwidth=hwidth, fill=color, stroke='none')
    print(f"  Level {level:+d}: y={height:.4f}")

# --- Draw rays from interior kinks ---
print("\nRays from interior kinks:")

total_rays = 0
for parent_level in range(MAX_LEVEL, MIN_LEVEL, -1):
    child_level = parent_level - 1
    child_spacing = m ** child_level
    child_height = child_spacing

    # Interior kinks: x = j * child_spacing where j % m != 0
    j = 1
    num_rays = 0
    while j * child_spacing < x_range:
        if j % m != 0:  # interior kink (not at parent boundary)
            kink_x = j * child_spacing + OFFSET
            scene.add_ray(complex(kink_x, child_height), kink_x,
                          hwidth=hwidth, fill=ray_color, stroke='none')
            num_rays += 1
        j += 1

    if num_rays > 0:
        print(f"  Level {child_level:+d}: {num_rays} rays")
        total_rays += num_rays

# --- Add two boundary rays at the largest layer ---
print("\nBoundary rays at largest layer:")
top_height = m ** MAX_LEVEL
top_spacing = top_height

scene.add_ray(complex(OFFSET, top_height), OFFSET,
              hwidth=hwidth, fill=ray_color, stroke='none')
print(f"  Ray from x={OFFSET} at y={top_height}")

first_kink_x = top_spacing + OFFSET
if first_kink_x < x_range + OFFSET:
    scene.add_ray(complex(first_kink_x, top_height), first_kink_x,
                  hwidth=hwidth, fill=ray_color, stroke='none')
    print(f"  Ray from x={first_kink_x} at y={top_height}")
    total_rays += 2
else:
    total_rays += 1

print(f"\nTotal: {MAX_LEVEL - MIN_LEVEL + 1} horocycles, {total_rays} rays")

# =============================================================================
# RENDER AND DISPLAY
# =============================================================================

svg = scene.render(width=WIDTH, height=HEIGHT, background='white')
display(SVG(svg.as_svg()))

# Save to folder output/
svg.save_svg(f'output/horocycle_scene_m{m}.svg')
print(f"Saved to output/horocycle_scene_m{m}.svg")

## 1b. Horocycle Tiling with 3-Coloring

Same structure as the horocycle scene, but now with filled tiles using the 3-coloring algorithm.
Each tile is a quadrilateral with horocycle edges (top/bottom) and geodesic edges (left/right).

In [ ]:
import math
import importlib
import hyperbolic_svg
importlib.reload(hyperbolic_svg)
from hyperbolic_svg import HyperbolicScene
from IPython.display import SVG, display

# =============================================================================
# PARAMETERS
# =============================================================================

m = 2               # Branching factor
TOP_LEVEL = 8       # Level of the single root tile
NUM_LAYERS = 14     # How many layers to descend from root
WIDTH = 600
HEIGHT = 600

# Derived
BOTTOM_LEVEL = TOP_LEVEL - NUM_LAYERS + 1
ROOT_WIDTH = m ** (TOP_LEVEL + 1)
OFFSET = -ROOT_WIDTH / 2  # Center the root polygon at x=0
x_range = ROOT_WIDTH

# 3-color palette
COLOR_PALETTE = [
    '#e6a4a4',  # soft red/pink (color 0)
    '#a4c9e6',  # soft blue (color 1)
    '#a4e6a4',  # soft green (color 2)
]

# Horocycle and ray styling
hwidth = 0.08
horocycle_color = '#333333'
ray_color = '#555555'

# =============================================================================
# 3-COLORING ALGORITHM
# =============================================================================

def get_available_colors(parent_color):
    return [c for c in [0, 1, 2] if c != parent_color]

def assign_colors_tree(m, num_levels):
    colors = {}
    top_level = num_levels - 1
    colors[(top_level, 0)] = 0
    
    for level in range(top_level - 1, -1, -1):
        parent_level = level + 1
        num_tiles = m ** (top_level - level)
        
        last_color = None
        for tile_idx in range(num_tiles):
            parent_idx = tile_idx // m
            child_pos = tile_idx % m
            
            parent_color = colors[(parent_level, parent_idx)]
            available = get_available_colors(parent_color)
            
            if child_pos == 0:
                start = available[0]
                if last_color is not None and start == last_color:
                    start = available[1]
                color = start
            else:
                prev = colors[(level, tile_idx - 1)]
                color = available[1] if prev == available[0] else available[0]
            
            colors[(level, tile_idx)] = color
            last_color = color
    
    return colors

def level_color(level, min_lvl, max_lvl):
    """Generate a color based on level (hue rotation)."""
    t = (level - min_lvl) / (max_lvl - min_lvl) if max_lvl != min_lvl else 0.5
    hue = t * 300
    if hue < 60:
        r, g, b = 255, int(hue * 255 / 60), 0
    elif hue < 120:
        r, g, b = int((120 - hue) * 255 / 60), 255, 0
    elif hue < 180:
        r, g, b = 0, 255, int((hue - 120) * 255 / 60)
    elif hue < 240:
        r, g, b = 0, int((240 - hue) * 255 / 60), 255
    else:
        r, g, b = int((hue - 240) * 255 / 60), 0, 255
    return f'#{r:02x}{g:02x}{b:02x}'

# =============================================================================
# BUILD THE HOROCYCLE TILING
# =============================================================================

scene = HyperbolicScene()

print(f"Horocycle tiling with 3-coloring")
print(f"m={m}, TOP_LEVEL={TOP_LEVEL}, NUM_LAYERS={NUM_LAYERS}")
print(f"Levels: {BOTTOM_LEVEL} to {TOP_LEVEL}")

colors = assign_colors_tree(m, NUM_LAYERS)

# --- Draw tiles first (background) ---
color_counts = {0: 0, 1: 0, 2: 0}

for level in range(BOTTOM_LEVEL, TOP_LEVEL + 1):
    tree_level = level - BOTTOM_LEVEL
    num_tiles = m ** (NUM_LAYERS - 1 - tree_level)
    
    y_bottom = m ** level
    y_top = m ** (level + 1)
    tile_width = y_top
    
    for j in range(num_tiles):
        key = (tree_level, j)
        if key not in colors:
            continue
        color_idx = colors[key]
        fill = COLOR_PALETTE[color_idx]
        color_counts[color_idx] += 1
        
        x_left = j * tile_width + OFFSET
        x_right = x_left + tile_width
        
        scene.add_horocycle_tile(x_left, x_right, y_bottom, y_top,
                                 fill=fill, stroke='none')

total = sum(color_counts.values())
print(f"Total tiles: {total}")

# --- Draw horocycles at each level ---
print("\nHorocycles:")
for level in range(BOTTOM_LEVEL, TOP_LEVEL + 1):
    height = m ** level
    color = level_color(level, BOTTOM_LEVEL, TOP_LEVEL)
    scene.add_horizontal_horocycle(height, hwidth=hwidth, fill=horocycle_color, stroke='none')
    print(f"  Level {level:+d}: y={height:.4f}")

# --- Draw rays from interior kinks ---
print("\nRays:")
total_rays = 0
for parent_level in range(TOP_LEVEL, BOTTOM_LEVEL, -1):
    child_level = parent_level - 1
    child_spacing = m ** child_level
    child_height = child_spacing
    
    j = 1
    num_rays = 0
    while j * child_spacing < x_range:
        if j % m != 0:
            kink_x = j * child_spacing + OFFSET
            scene.add_ray(complex(kink_x, child_height), kink_x,
                          hwidth=hwidth, fill=ray_color, stroke='none')
            num_rays += 1
        j += 1
    
    if num_rays > 0:
        print(f"  Level {child_level:+d}: {num_rays} rays")
        total_rays += num_rays

# Boundary rays at top level
top_height = m ** TOP_LEVEL
scene.add_ray(complex(OFFSET, top_height), OFFSET,
              hwidth=hwidth, fill=ray_color, stroke='none')
first_kink_x = m ** TOP_LEVEL + OFFSET
if first_kink_x < x_range + OFFSET:
    scene.add_ray(complex(first_kink_x, top_height), first_kink_x,
                  hwidth=hwidth, fill=ray_color, stroke='none')
    total_rays += 2
else:
    total_rays += 1

print(f"\nTotal: {total} tiles, {TOP_LEVEL - BOTTOM_LEVEL + 1} horocycles, {total_rays} rays")

svg = scene.render(width=WIDTH, height=HEIGHT, background='white')
display(SVG(svg.as_svg()))

# Save to folder output/
svg.save_svg(f'output/3_colored_tiling_horo_m{m}.svg')
print(f"Saved to output/3_colored_tiling_horo_m{m}.svg")

## 2. Bumps Scene with Polygon

Horizontal "bumps" (geodesic segments) at multiple scale levels, with rays from interior kinks connecting the hierarchy. A single polygon is placed to demonstrate the tile fit.

In [ ]:
import math
from hyperbolic_svg import HyperbolicScene, make_hyperbolic_polygon_from_params
from IPython.display import SVG, display

# =============================================================================
# PARAMETERS - Adjust these to customize the scene
# =============================================================================

m = 2              # Scaling factor (branching ratio): each bump contains m child bumps
x_range = 1000       # Horizontal extent of the scene (before offset)
OFFSET = -x_range/2 - 1/8       # Translate scene by this real value (shifts away from ideal point 0)
MIN_LEVEL = -5     # Lowest level (finest detail, closest to boundary)
MAX_LEVEL = 9      # Highest level (coarsest detail, closest to center)
hwidth = 0.08      # Hyperbolic line width (same for all elements)
ray_color = '#555555'  # Color for rays from interior kinks
WIDTH = 600        # SVG width in pixels
HEIGHT = 600       # SVG height in pixels

# Polygon parameters
SHOW_POLYGON = True        # Whether to show the polygon
POLYGON_LEVEL = 0          # Which level to place the polygon at (bottom edge at y = m^level)
polygon_fill = '#a8d5ba'   # Soft sage green (fill only, no stroke)

# =============================================================================
# COLOR FUNCTION
# =============================================================================

def level_color(level, min_lvl, max_lvl):
    """Generate a color based on level (hue rotation from red to purple)."""
    t = (level - min_lvl) / (max_lvl - min_lvl) if max_lvl != min_lvl else 0.5
    hue = t * 300  # 0 = red, 60 = yellow, 120 = green, 180 = cyan, 240 = blue, 300 = magenta
    if hue < 60:
        r, g, b = 255, int(hue * 255 / 60), 0
    elif hue < 120:
        r, g, b = int((120 - hue) * 255 / 60), 255, 0
    elif hue < 180:
        r, g, b = 0, 255, int((hue - 120) * 255 / 60)
    elif hue < 240:
        r, g, b = 0, int((240 - hue) * 255 / 60), 255
    else:
        r, g, b = int((hue - 240) * 255 / 60), 0, 255
    return f'#{r:02x}{g:02x}{b:02x}'

# =============================================================================
# BUILD THE SCENE
# =============================================================================

scene = HyperbolicScene()

# --- Draw bumps at each level ---
print(f"Building scene with m={m}, levels {MIN_LEVEL} to {MAX_LEVEL}, x in [{OFFSET}, {OFFSET + x_range}]")
print("\nBumps:")

for level in range(MIN_LEVEL, MAX_LEVEL + 1):
    scale = m ** level
    spacing = scale
    height = scale
    num_segments = math.ceil(x_range / spacing)

    # Generate points with OFFSET applied
    points = []
    for i in range(num_segments + 1):
        x = min(i * spacing, x_range)
        points.append(complex(x + OFFSET, height))

    # Remove duplicate endpoint if clamped
    if len(points) > 1 and points[-1] == points[-2]:
        points.pop()

    # Add segments
    color = level_color(level, MIN_LEVEL, MAX_LEVEL)
    for i in range(len(points) - 1):
        scene.add_segment(points[i], points[i+1], hwidth=hwidth, fill=color, stroke='none')

    print(f"  Level {level:+d}: y={height:.4f}, {len(points)-1} segments")

# --- Draw rays from interior kinks ---
print("\nRays from interior kinks:")

total_rays = 0
for parent_level in range(MAX_LEVEL, MIN_LEVEL, -1):
    child_level = parent_level - 1
    child_spacing = m ** child_level
    child_height = child_spacing

    # Interior kinks: x = j * child_spacing where j % m != 0
    j = 1
    num_rays = 0
    while j * child_spacing < x_range:
        if j % m != 0:  # interior kink (not at parent boundary)
            kink_x = j * child_spacing + OFFSET
            scene.add_ray(complex(kink_x, child_height), kink_x,
                          hwidth=hwidth, fill=ray_color, stroke='none')
            num_rays += 1
        j += 1

    if num_rays > 0:
        print(f"  Level {child_level:+d}: {num_rays} rays")
        total_rays += num_rays

# --- Add two boundary rays at the largest layer ---
print("\nBoundary rays at largest layer:")
top_height = m ** MAX_LEVEL
top_spacing = top_height

# Ray from x = 0 (left boundary)
scene.add_ray(complex(OFFSET, top_height), OFFSET,
              hwidth=hwidth, fill=ray_color, stroke='none')
print(f"  Ray from x={OFFSET} at y={top_height}")

# Ray from first kink (x = spacing of largest layer)
first_kink_x = top_spacing + OFFSET
if first_kink_x < x_range + OFFSET:
    scene.add_ray(complex(first_kink_x, top_height), first_kink_x,
                  hwidth=hwidth, fill=ray_color, stroke='none')
    print(f"  Ray from x={first_kink_x} at y={top_height}")
    total_rays += 2
else:
    total_rays += 1

print(f"\nTotal: {MAX_LEVEL - MIN_LEVEL + 1} levels, {total_rays} rays")

# --- Add polygon LAST (so it's on top) ---
if SHOW_POLYGON:
    print("\nPolygon (rendered on top):")
    N = m + 3
    a = m ** POLYGON_LEVEL
    base_vertices = make_hyperbolic_polygon_from_params(N=N, a=a)
    scale_factor = m ** POLYGON_LEVEL
    vertices = [complex(v.real * scale_factor + OFFSET, v.imag * scale_factor) for v in base_vertices]
    
    # No hwidth = solid fill (not thick outline)
    scene.add_polygon(vertices, fill=polygon_fill, stroke='none')
    
    print(f"  N={N}, placed at level {POLYGON_LEVEL}")
    print(f"  Spans: x=[{min(v.real for v in vertices):.1f}, {max(v.real for v in vertices):.1f}], y=[{min(v.imag for v in vertices):.1f}, {max(v.imag for v in vertices):.1f}]")

# =============================================================================
# RENDER AND DISPLAY
# =============================================================================

svg = scene.render(width=WIDTH, height=HEIGHT, background='white')
display(SVG(svg.as_svg()))

# Save to folder output/
svg.save_svg(f'output/hyperbolicPolygon_scene_m{m}.svg')
print(f"Saved to output/hyperbolicPolygon_scene_m{m}.svg")

## 3. Debug: Single Polygon

Render just the polygon to verify its shape and placement.

In [ ]:
# Debug cell: Render just the polygon with hyperbolic line width
from hyperbolic_svg import HyperbolicScene, make_hyperbolic_polygon_from_params
from IPython.display import SVG, display

# Parameters
m = 2
OFFSET = 0.2
POLYGON_LEVEL = 0
polygon_fill = '#a8d5ba'
hwidth = 0.05  # Hyperbolic line width
boundary_color = '#333333'

# Get polygon vertices
N = m + 3
scale_factor = m ** POLYGON_LEVEL
base_vertices = make_hyperbolic_polygon_from_params(N=N, a=1)
vertices = [complex(v.real * scale_factor + OFFSET, v.imag * scale_factor) for v in base_vertices]

print(f"Polygon parameters: N={N}, scale={scale_factor}, OFFSET={OFFSET}")
print(f"hwidth={hwidth}")
print(f"Vertices:")
for v in vertices:
    print(f"  ({v.real:.2f}, {v.imag:.2f})")

# Create a scene with the polygon using TWO-PASS approach:
# 1. Fill only (no hwidth)
# 2. Hyperbolic boundary only (hwidth, no interior fill)
test_scene = HyperbolicScene()
test_scene.add_polygon(vertices, fill=polygon_fill, stroke='none')  # Fill
test_scene.add_polygon(vertices, hwidth=hwidth, fill=boundary_color, stroke='none')  # Boundary

svg = test_scene.render(width=400, height=400, background='white')
display(SVG(svg.as_svg()))

## 4. Full Tiling with Bumps

Tile the entire scene with polygons while keeping the bump and ray structure visible.

In [ ]:
# Optional: Save to file
svg.save_svg(f'output/bumps_m{m}.svg')
print(f"Saved to output/bumps_m{m}.svg")

## 5. Pure Polygon Tiling

Clean tiling with only polygons (no bumps or rays), with thin boundaries to show tile edges.

## 6. 3-Coloring Algorithm

The polygon tiling is 3-colorable. Below we work through the algorithm by hand for m=2 and m=3 with 3 layers each.

### 3-Coloring Algorithm

**Goal:** Assign colors {0, 1, 2} to tiles such that no two adjacent tiles share the same color.

**Adjacency:** Two tiles are adjacent if they share an edge:
- Siblings (children of the same parent) are adjacent horizontally
- Parent-child tiles share a horizontal edge
- At the same level, tiles from different parent groups can be adjacent at group boundaries

**Algorithm:**
1. Start at the top level. The root tile gets color 0.
2. For each tile with color c, its m children alternate between the two colors in {0, 1, 2} \ {c}.
3. **Boundary fix:** When starting a new group, if the first child's color would match the last leaf of the left neighbor, switch to the other available color.

---

### Example: m = 2, 3 Layers

**Level 2:** `[0]`

**Level 1:** Children of 0 use {1, 2}, start with 1 → `[1, 2]`

**Level 0 (without fix):**
- Under parent 1: use {0, 2}, start with 2 → `[2, 0]`, last leaf = 0
- Under parent 2: use {0, 1}, start with 0 → `[0, 1]`
- Result: `[2, 0, 0, 1]` — **CONFLICT** (tiles 1 and 2 both = 0)

**Level 0 (with fix):**
- Under parent 1: use {0, 2}, start with 2 → `[2, 0]`, last leaf = 0
- Under parent 2: use {0, 1}, would start 0, but left neighbor ends with 0 → **switch to 1** → `[1, 0]`
- Result: `[2, 0, 1, 0]` ✓

```
Level 2:     [    0    ]
Level 1:     [ 1 ][ 2 ]
Level 0:     [2][0][1][0]
                 ↑
            fixed: 0→1
```

---

### Example: m = 3, 3 Layers

**Level 2:** `[0]`

**Level 1:** Children of 0 use {1, 2}, start with 1, alternate → `[1, 2, 1]`

**Level 0 (with fix):**
- Under parent 1 (idx 0): use {0, 2}, no left neighbor, start 0 → `[0, 2, 0]`, last = 0
- Under parent 2 (idx 1): use {0, 1}, would start 0, left ends 0 → **switch to 1** → `[1, 0, 1]`, last = 1
- Under parent 1 (idx 2): use {0, 2}, would start 0, left ends 1 → no conflict → `[0, 2, 0]`, last = 0

Result: `[0, 2, 0, 1, 0, 1, 0, 2, 0]` ✓

```
Level 2:     [         0         ]
Level 1:     [  1  ][  2  ][  1  ]
Level 0:     [0][2][0][1][0][1][0][2][0]
                   ↑     ↑
              fixed    no fix (1≠0)
```

Check boundaries:
- Tile 2 (0) vs Tile 3 (1): different ✓
- Tile 5 (1) vs Tile 6 (0): different ✓

---

### Summary of Algorithm

```
for each tile at level k:
    parent_color = tile's parent color
    available = {0, 1, 2} - {parent_color}  # two colors
    
    # Determine starting color
    if left_neighbor_exists:
        left_last = last leaf color of left neighbor's subtree
        start = available[0]
        if start == left_last:
            start = available[1]  # switch to avoid conflict
    else:
        start = available[0]
    
    # Alternate between the two available colors
    for i in range(m):
        child_color = available[(available.index(start) + i) % 2]
```

In [ ]:
# 3-Coloring Implementation
from hyperbolic_svg import HyperbolicScene, make_hyperbolic_polygon_from_params
from IPython.display import SVG, display

# =============================================================================
# PARAMETERS
# =============================================================================
m = 3
x_range = 100
OFFSET = -50
MIN_LEVEL = -4
MAX_LEVEL = 5
WIDTH = 600
HEIGHT = 600

# 3-color palette (soft, distinct colors)
COLOR_PALETTE = [
    '#e6a4a4',  # soft red/pink (color 0)
    '#a4c9e6',  # soft blue (color 1)
    '#a4e6a4',  # soft green (color 2)
]

stroke_color = "#000000FF"
stroke_width = 0.003

# =============================================================================
# 3-COLORING ALGORITHM
# =============================================================================

def get_available_colors(parent_color):
    """Return the two colors available for children (excluding parent color)."""
    return [c for c in [0, 1, 2] if c != parent_color]

def assign_colors(m, num_levels):
    """
    Assign 3-colors to all tiles in the hierarchy.
    Returns dict: (level, tile_index) -> color
    """
    colors = {}
    
    # Top level: single tile with color 0
    top_level = num_levels - 1
    colors[(top_level, 0)] = 0
    
    # Process each level from top to bottom
    for level in range(top_level - 1, -1, -1):
        parent_level = level + 1
        num_tiles_at_level = m ** (top_level - level)
        
        last_leaf_color = None  # Track last assigned color for boundary fix
        
        for tile_idx in range(num_tiles_at_level):
            parent_idx = tile_idx // m
            child_position = tile_idx % m  # 0 to m-1 within parent's children
            
            parent_color = colors[(parent_level, parent_idx)]
            available = get_available_colors(parent_color)
            
            if child_position == 0:
                # First child of this parent - check for boundary conflict
                start_color = available[0]
                if last_leaf_color is not None and start_color == last_leaf_color:
                    start_color = available[1]  # Switch to avoid conflict
                color = start_color
            else:
                # Alternate from previous sibling
                prev_color = colors[(level, tile_idx - 1)]
                # Pick the other available color
                color = available[1] if prev_color == available[0] else available[0]
            
            colors[(level, tile_idx)] = color
            last_leaf_color = color
    
    return colors

# =============================================================================
# BUILD THE 3-COLORED TILING
# =============================================================================

scene = HyperbolicScene()

# Calculate how many complete tiles we can fit
# We'll work with a fixed hierarchy depth for coloring
num_levels = MAX_LEVEL - MIN_LEVEL

print(f"3-Coloring tiling with m={m}")
print(f"Levels {MIN_LEVEL} to {MAX_LEVEL-1}, x in [{OFFSET}, {OFFSET + x_range}]")
print(f"Colors: {COLOR_PALETTE}")
print()

# For each level, calculate tiles and assign colors
total_tiles = 0
color_counts = {0: 0, 1: 0, 2: 0}

for level in range(MIN_LEVEL, MAX_LEVEL):
    scale = m ** level
    tile_width = m ** (level + 1)
    num_tiles = int(x_range / tile_width)
    
    if num_tiles == 0:
        continue
    
    # Get base polygon vertices
    base_vertices = make_hyperbolic_polygon_from_params(N=m+3, a=1)
    
    # For coloring: map this level to the hierarchy
    # level MIN_LEVEL is the bottom (finest), MAX_LEVEL-1 is top (coarsest)
    hierarchy_level = level - MIN_LEVEL
    hierarchy_depth = MAX_LEVEL - MIN_LEVEL
    
    # Assign colors for this level's tiles
    level_colors = assign_colors(m, hierarchy_depth)
    
    for j in range(num_tiles):
        # Get color from the 3-coloring
        color_idx = level_colors.get((hierarchy_level, j), 0)
        fill = COLOR_PALETTE[color_idx]
        color_counts[color_idx] += 1
        
        # Create tile vertices
        vertices = [complex(v.real * scale + j * tile_width + OFFSET, v.imag * scale) 
                    for v in base_vertices]
        scene.add_polygon(vertices, fill=fill, stroke=stroke_color, stroke_width=stroke_width)
        total_tiles += 1
    
    print(f"  Level {level:+d}: {num_tiles} tiles")

print(f"\nTotal: {total_tiles} tiles")
print(f"Color distribution: red={color_counts[0]}, blue={color_counts[1]}, green={color_counts[2]}")

# Render
svg = scene.render(width=WIDTH, height=HEIGHT, background='white')
display(SVG(svg.as_svg()))

# Save to folder output/
svg.save_svg(f'output/3_colored_tiling_non-hwidth_m{m}_offset{OFFSET}.svg')
print(f"Saved to output/3_colored_tiling_non-hwidth_m{m}_offset{OFFSET}.svg")

In [ ]:
# 3-Coloring with Hyperbolic Line Width (FIXED: single root tree structure)
from hyperbolic_svg import HyperbolicScene, make_hyperbolic_polygon_from_params
from IPython.display import SVG, display

# =============================================================================
# PARAMETERS
# =============================================================================
m = 2               # Branching factor
TOP_LEVEL = 8       # Level of the single root tile (scaled by m^TOP_LEVEL)
NUM_LAYERS = 14     # How many layers to descend from root
WIDTH = 600
HEIGHT = 600

# Derived
BOTTOM_LEVEL = TOP_LEVEL - NUM_LAYERS + 1
ROOT_WIDTH = m ** (TOP_LEVEL + 1)
OFFSET = -ROOT_WIDTH / 2 - 30 + 1/8  # Center the root polygon at x=0

print(f"DEBUG: BOTTOM_LEVEL={BOTTOM_LEVEL}, TOP_LEVEL={TOP_LEVEL}")
print(f"DEBUG: ROOT_WIDTH={ROOT_WIDTH}, OFFSET={OFFSET}")

# 3-color palette
COLOR_PALETTE = [
    '#e6a4a4',  # soft red/pink (color 0)
    '#a4c9e6',  # soft blue (color 1)
    '#a4e6a4',  # soft green (color 2)
]

# Hyperbolic line width for boundaries
hwidth = 0.02
boundary_color = '#333333'

# =============================================================================
# 3-COLORING ALGORITHM
# =============================================================================

def get_available_colors(parent_color):
    """Return the two colors available for children (excluding parent color)."""
    return [c for c in [0, 1, 2] if c != parent_color]

def assign_colors_tree(m, num_levels):
    """
    Assign 3-colors to a single-root tree.
    Level 0 = bottom (finest), level num_levels-1 = top (root)
    Returns dict: (level, tile_index) -> color
    """
    colors = {}
    
    # Root gets color 0
    top_level = num_levels - 1
    colors[(top_level, 0)] = 0
    print(f"DEBUG: assign_colors top_level={top_level}, num_levels={num_levels}")
    
    # Process each level from top to bottom
    for level in range(top_level - 1, -1, -1):
        parent_level = level + 1
        num_tiles = m ** (top_level - level)
        
        last_color = None
        
        for tile_idx in range(num_tiles):
            parent_idx = tile_idx // m
            child_pos = tile_idx % m
            
            parent_color = colors[(parent_level, parent_idx)]
            available = get_available_colors(parent_color)
            
            if child_pos == 0:
                # First child - check boundary conflict
                start = available[0]
                if last_color is not None and start == last_color:
                    start = available[1]
                color = start
            else:
                # Alternate from previous sibling
                prev = colors[(level, tile_idx - 1)]
                color = available[1] if prev == available[0] else available[0]
            
            colors[(level, tile_idx)] = color
            last_color = color
    
    print(f"DEBUG: colors dict has {len(colors)} entries")
    return colors

# =============================================================================
# BUILD THE TILING (single root tree)
# =============================================================================

scene = HyperbolicScene()

print(f"3-Coloring with single root tree")
print(f"m={m}, TOP_LEVEL={TOP_LEVEL}, NUM_LAYERS={NUM_LAYERS}")
print(f"Levels: {BOTTOM_LEVEL} to {TOP_LEVEL}")
print(f"Root tile width: {ROOT_WIDTH}, OFFSET: {OFFSET} (centered)")
print()

# Assign colors for the tree
colors = assign_colors_tree(m, NUM_LAYERS)

# Collect all tiles
all_tiles = []
color_counts = {0: 0, 1: 0, 2: 0}

for level in range(BOTTOM_LEVEL, TOP_LEVEL + 1):
    # Map to tree level (0 = bottom of tree, NUM_LAYERS-1 = root)
    tree_level = level - BOTTOM_LEVEL
    num_tiles = m ** (NUM_LAYERS - 1 - tree_level)
    
    scale = m ** level
    tile_width = m ** (level + 1)
    
    base_vertices = make_hyperbolic_polygon_from_params(N=m+3, a=1)
    
    print(f"DEBUG: level={level}, tree_level={tree_level}, num_tiles={num_tiles}, scale={scale}")
    
    for j in range(num_tiles):
        key = (tree_level, j)
        if key not in colors:
            print(f"DEBUG: KEY NOT FOUND: {key}")
            continue
        color_idx = colors[key]
        fill = COLOR_PALETTE[color_idx]
        color_counts[color_idx] += 1
        
        vertices = [complex(v.real * scale + j * tile_width + OFFSET, v.imag * scale) 
                    for v in base_vertices]
        all_tiles.append((vertices, fill))
    
    print(f"  Level {level:+d} (tree level {tree_level}): {num_tiles} tiles")

print(f"\nDEBUG: all_tiles has {len(all_tiles)} entries")

# TWO-PASS RENDERING
# Pass 1: fills
for vertices, fill in all_tiles:
    scene.add_polygon(vertices, fill=fill, stroke='none')

# Pass 2: hyperbolic boundaries
for vertices, fill in all_tiles:
    scene.add_polygon(vertices, hwidth=hwidth, fill=boundary_color, stroke='none')

total = sum(color_counts.values())
print(f"\nTotal: {total} tiles")
print(f"Colors: red={color_counts[0]}, blue={color_counts[1]}, green={color_counts[2]}")

svg = scene.render(width=WIDTH, height=HEIGHT, background='white')
print(f"DEBUG: svg rendered")
display(SVG(svg.as_svg()))

# Save to folder output/
svg.save_svg(f'output/3_colored_tiling_hwidth_m{m}_offset{OFFSET}.svg')
print(f"Saved to output/3_colored_tiling_hwidth_m{m}_offset{OFFSET}.svg")

In [ ]:
# Pure polygon tiling (m=2) - no bumps or rays, just the tiles with thin boundaries
from hyperbolic_svg import HyperbolicScene, make_hyperbolic_polygon_from_params
from IPython.display import SVG, display
import colorsys

# =============================================================================
# PARAMETERS
# =============================================================================
m = 2
x_range = 200
OFFSET = -20
MIN_LEVEL = -5
MAX_LEVEL = 5
WIDTH = 600
HEIGHT = 600

# Tile styling
stroke_color = "#33333300"   # Dark gray boundary
stroke_width = 0.005       # Thin boundary

def tile_color(level, min_lvl, max_lvl):
    """Soft pastel color for each level."""
    t = (level - min_lvl) / (max_lvl - min_lvl) if max_lvl != min_lvl else 0.5
    h = t * 0.8  # hue from red to purple
    s = 0.35     # low saturation
    l = 0.80     # high lightness for pastel
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

# =============================================================================
# BUILD TILED SCENE
# =============================================================================
scene = HyperbolicScene()

print(f"Pure polygon tiling with m={m}")
print(f"Levels {MIN_LEVEL} to {MAX_LEVEL}, x in [{OFFSET}, {OFFSET + x_range}]")
print()

total_tiles = 0
for level in range(MIN_LEVEL, MAX_LEVEL):
    scale = m ** level
    tile_width = m ** (level + 1)
    num_tiles = int(x_range / tile_width)
    
    if num_tiles == 0:
        continue
    
    base_vertices = make_hyperbolic_polygon_from_params(N=m+3, a=1)
    fill = tile_color(level, MIN_LEVEL, MAX_LEVEL)
    
    for j in range(num_tiles):
        vertices = [complex(v.real * scale + j * tile_width + OFFSET, v.imag * scale) 
                    for v in base_vertices]
        scene.add_polygon(vertices, fill=fill, stroke=stroke_color, stroke_width=stroke_width)
        total_tiles += 1
    
    print(f"  Level {level:+d} to {level+1:+d}: {num_tiles} tiles, color={fill}")

print(f"\nTotal: {total_tiles} tiles")

svg = scene.render(width=WIDTH, height=HEIGHT, background='white')
display(SVG(svg.as_svg()))

# Save to folder output/
svg.save_svg(f'output/pure_polygon_tiling_m{m}_offset{OFFSET}.svg')
print(f"Saved to output/pure_polygon_tiling_m{m}_offset{OFFSET}.svg")

---

# Upper Half-Plane Visualizations

The same scenes rendered in the Upper Half-Plane model with natural (linear) coordinates.

**Key differences from Poincaré disc:**
- Horizontal horocycles → horizontal lines/strips
- Vertical geodesics with hwidth → wedge shapes (width proportional to height)
- Tiles → rectangles (height varies with scale level)

In UHP, the self-similar structure is visible through the scaling: tiles at level k have height m^(k+1) - m^k.

In [ ]:
# UHP: Horocycle Scene
import math
import importlib
import hyperbolic_svg
importlib.reload(hyperbolic_svg)
from hyperbolic_svg import HyperbolicScene
from IPython.display import SVG, display

# =============================================================================
# PARAMETERS (same as disc version)
# =============================================================================

m = 2
x_range = 1000
OFFSET = -x_range/2 - 1/8
MIN_LEVEL = -5
MAX_LEVEL = 9
ray_color = '#555555'

def level_color(level, min_lvl, max_lvl):
    t = (level - min_lvl) / (max_lvl - min_lvl) if max_lvl != min_lvl else 0.5
    hue = t * 300
    if hue < 60:
        r, g, b = 255, int(hue * 255 / 60), 0
    elif hue < 120:
        r, g, b = int((120 - hue) * 255 / 60), 255, 0
    elif hue < 180:
        r, g, b = 0, 255, int((hue - 120) * 255 / 60)
    elif hue < 240:
        r, g, b = 0, int((240 - hue) * 255 / 60), 255
    else:
        r, g, b = int((hue - 240) * 255 / 60), 0, 255
    return f'#{r:02x}{g:02x}{b:02x}'

# =============================================================================
# BUILD THE SCENE
# =============================================================================

scene = HyperbolicScene()

# Horocycles
for level in range(MIN_LEVEL, MAX_LEVEL + 1):
    height = m ** level
    color = level_color(level, MIN_LEVEL, MAX_LEVEL)
    scene.add_horizontal_horocycle(height, fill=color, stroke='none')

# Rays
for parent_level in range(MAX_LEVEL, MIN_LEVEL, -1):
    child_level = parent_level - 1
    child_spacing = m ** child_level
    child_height = child_spacing
    j = 1
    while j * child_spacing < x_range:
        if j % m != 0:
            kink_x = j * child_spacing + OFFSET
            scene.add_ray(complex(kink_x, child_height), kink_x, fill=ray_color, stroke='none')
        j += 1

# Boundary rays
top_height = m ** MAX_LEVEL
scene.add_ray(complex(OFFSET, top_height), OFFSET, fill=ray_color, stroke='none')
scene.add_ray(complex(m ** MAX_LEVEL + OFFSET, top_height), m ** MAX_LEVEL + OFFSET,
              fill=ray_color, stroke='none')

# =============================================================================
# RENDER IN UHP (simple Euclidean strokes)
# =============================================================================

print("Rendering in Upper Half-Plane (constant line width)...")

svg_uhp = scene.render_uhp(
    width=800, height=400,
    x_min=OFFSET, x_max=OFFSET + x_range,
    y_min=m ** MIN_LEVEL, y_max=m ** (MAX_LEVEL + 1),
    background='white',
    stroke_width=1.0
)

display(SVG(svg_uhp.as_svg()))
print(f"Viewport: x=[{OFFSET:.1f}, {OFFSET + x_range:.1f}], y=[{m**MIN_LEVEL}, {m**(MAX_LEVEL+1)}]")

In [ ]:
# UHP: Horocycle Tiling with 3-Coloring
import math
import importlib
import hyperbolic_svg
importlib.reload(hyperbolic_svg)
from hyperbolic_svg import HyperbolicScene
from IPython.display import SVG, display

# =============================================================================
# PARAMETERS
# =============================================================================

m = 2
TOP_LEVEL = 8
NUM_LAYERS = 14
BOTTOM_LEVEL = TOP_LEVEL - NUM_LAYERS + 1
ROOT_WIDTH = m ** (TOP_LEVEL + 1)
OFFSET = -ROOT_WIDTH / 2
x_range = ROOT_WIDTH

COLOR_PALETTE = ['#e6a4a4', '#a4c9e6', '#a4e6a4']
horocycle_color = '#333333'
ray_color = '#555555'

# =============================================================================
# 3-COLORING
# =============================================================================

def get_available_colors(parent_color):
    return [c for c in [0, 1, 2] if c != parent_color]

def assign_colors_tree(m, num_levels):
    colors = {}
    top_level = num_levels - 1
    colors[(top_level, 0)] = 0
    for level in range(top_level - 1, -1, -1):
        parent_level = level + 1
        num_tiles = m ** (top_level - level)
        last_color = None
        for tile_idx in range(num_tiles):
            parent_idx = tile_idx // m
            child_pos = tile_idx % m
            parent_color = colors[(parent_level, parent_idx)]
            available = get_available_colors(parent_color)
            if child_pos == 0:
                start = available[0]
                if last_color is not None and start == last_color:
                    start = available[1]
                color = start
            else:
                prev = colors[(level, tile_idx - 1)]
                color = available[1] if prev == available[0] else available[0]
            colors[(level, tile_idx)] = color
            last_color = color
    return colors

# =============================================================================
# BUILD SCENE
# =============================================================================

scene = HyperbolicScene()
colors = assign_colors_tree(m, NUM_LAYERS)

# Tiles
for level in range(BOTTOM_LEVEL, TOP_LEVEL + 1):
    tree_level = level - BOTTOM_LEVEL
    num_tiles = m ** (NUM_LAYERS - 1 - tree_level)
    y_bottom = m ** level
    y_top = m ** (level + 1)
    tile_width = y_top
    for j in range(num_tiles):
        key = (tree_level, j)
        if key not in colors:
            continue
        fill = COLOR_PALETTE[colors[key]]
        x_left = j * tile_width + OFFSET
        x_right = x_left + tile_width
        scene.add_horocycle_tile(x_left, x_right, y_bottom, y_top, fill=fill, stroke='none')

# Horocycles
for level in range(BOTTOM_LEVEL, TOP_LEVEL + 1):
    scene.add_horizontal_horocycle(m ** level, fill=horocycle_color, stroke='none')

# Rays
for parent_level in range(TOP_LEVEL, BOTTOM_LEVEL, -1):
    child_level = parent_level - 1
    child_spacing = m ** child_level
    child_height = child_spacing
    j = 1
    while j * child_spacing < x_range:
        if j % m != 0:
            kink_x = j * child_spacing + OFFSET
            scene.add_ray(complex(kink_x, child_height), kink_x, fill=ray_color, stroke='none')
        j += 1

top_height = m ** TOP_LEVEL
scene.add_ray(complex(OFFSET, top_height), OFFSET, fill=ray_color, stroke='none')
scene.add_ray(complex(m ** TOP_LEVEL + OFFSET, top_height), m ** TOP_LEVEL + OFFSET,
              fill=ray_color, stroke='none')

# =============================================================================
# RENDER UHP (simple Euclidean strokes)
# =============================================================================

print("Rendering 3-colored tiling in Upper Half-Plane (constant line width)...")

svg_uhp = scene.render_uhp(
    width=800, height=400,
    x_min=OFFSET, x_max=OFFSET + x_range,
    y_min=m ** BOTTOM_LEVEL, y_max=m ** (TOP_LEVEL + 1),
    background='white',
    stroke_width=1.0
)

display(SVG(svg_uhp.as_svg()))

## Convert SVG Files to PNG

Convert all generated SVG files to PNG format and save them in the `output/png/` subfolder.

In [ ]:
import os
import glob
from pathlib import Path
import cairosvg

# Create output/png directory if it doesn't exist
png_dir = Path('output/png')
png_dir.mkdir(parents=True, exist_ok=True)

# Find all SVG files in the output directory
svg_files = glob.glob('output/*.svg')

print(f"Found {len(svg_files)} SVG files to convert:")
for svg_path in sorted(svg_files):
    svg_filename = Path(svg_path).name
    png_filename = svg_filename.replace('.svg', '.png')
    png_path = png_dir / png_filename
    
    print(f"  Converting {svg_filename} -> png/{png_filename}")
    cairosvg.svg2png(url=svg_path, write_to=str(png_path), scale=2.0)

print(f"\nAll files saved to {png_dir}/")
print(f"Total: {len(svg_files)} PNG files created")